# 1.0 Exploración de viajes MiBici (2022-01 a 2026-09)

Notebook solo de exploración. La lógica final vive en el módulo `mibici` (ingesta en `dataset.py`, variables en `features.py`).

Requisito: haber corrido `python -m mibici.dataset` y `python -m mibici.features`.

In [ ]:
import json

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from mibici.config import DATA_MANIFEST, PANEL_PARQUET, STATIONS_PARQUET, TRIPS_INTERIM_DIR

TRIPS = str(TRIPS_INTERIM_DIR / 'trips_*.parquet')
manifest = json.loads(DATA_MANIFEST.read_text())
print('Versión de datos:', manifest['data_version'])
manifest['totals']

## Calidad de la ingesta por mes
Codificación, formato de fecha y registros eliminados en la limpieza.

In [ ]:
files = pd.DataFrame(manifest['files'])
files[['month', 'encoding', 'date_format', 'rows_raw', 'rows_clean', 'dropped_duration', 'dropped_failed_dock', 'dropped_unknown_station']]

## Viajes por mes

In [ ]:
monthly = duckdb.sql(f"SELECT date_trunc('month', inicio) AS mes, count(*) AS viajes FROM read_parquet('{TRIPS}') GROUP BY 1 ORDER BY 1").df()
ax = monthly.plot(x='mes', y='viajes', figsize=(10, 3), legend=False, title='Viajes por mes')
ax.set_ylabel('viajes');

## Perfil horario: día hábil vs. fin de semana
Motiva las variables de calendario de P1 y los perfiles de P3.

In [ ]:
hourly = duckdb.sql(f"""SELECT hour(inicio) AS hora, dayofweek(inicio) IN (0, 6) AS fin_de_semana, count(*) AS viajes
FROM read_parquet('{TRIPS}') GROUP BY ALL""").df()
hourly.pivot(index='hora', columns='fin_de_semana', values='viajes').rename(columns={False: 'hábil', True: 'fin de semana'}).plot(figsize=(8, 3), title='Viajes por hora de inicio');

## Desbalance por estación (P2)
Distribución del flujo neto horario relativo a la capacidad y prevalencia de la etiqueta proxy.

In [ ]:
duckdb.sql(f"SELECT p2_label, count(*) AS horas, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct FROM read_parquet('{PANEL_PARQUET}') GROUP BY 1").df()

## Estaciones en el mapa

In [ ]:
st = pd.read_parquet(STATIONS_PARQUET)
fig, ax = plt.subplots(figsize=(6, 6))
for m, g in st.groupby('municipio'):
    ax.scatter(g['lon'], g['lat'], s=8, label=m)
ax.legend(); ax.set_title('Estaciones por municipio (nomenclatura 2026-09)');